# paper_sim_replay_v1

This production card validates the v1 paper/sim adapter boundary. It records target-position intent only and does not send real broker orders.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

root = Path.cwd().resolve()
while not (root / 'AGENTS.md').exists():
    if root.parent == root:
        raise RuntimeError('repo root not found')
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.api import PaperBroker

card_dir = root / 'production' / 'paper_sim_replay_v1'
card_dir.mkdir(parents=True, exist_ok=True)

broker = PaperBroker(cash=10_000)
order = broker.set_target_position('SOXL', target_percent=0.25, price=50.0)
order_intents = pd.DataFrame([{**order, 'price': 50.0}])
positions = pd.DataFrame([{'symbol': symbol, 'quantity': quantity} for symbol, quantity in broker.positions.items()])
order_intents.to_csv(card_dir / 'order_intents.csv', index=False)
positions.to_csv(card_dir / 'positions.csv', index=False)

conclusion = '''# 中文结论

- 做了什么：验证 production paper/sim adapter contract，只记录 target-position intent。
- 核心指标：SOXL target_percent=0.25，paper quantity=50，cash basis=10000，price=50。
- regime 差异：本卡不做 regime 选择，只验证执行边界。
- 是否晋升：不代表策略晋升；这是 v1 paper/sim 执行边界卡。
- 风险：没有连接真实 broker，没有真实订单、成交、回报或账户状态。
- 下一步：接入 production strategy notebook 的固定参数、风险说明和 replay 输入输出。
'''
(card_dir / 'conclusion.md').write_text(conclusion, encoding='utf-8')

summary = {
    'strategy': 'paper_sim_replay_v1',
    'execution_mode': 'paper_sim',
    'real_broker_orders': False,
    'artifacts': ['order_intents.csv', 'positions.csv', 'conclusion.md'],
}
(card_dir / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
summary


{'strategy': 'paper_sim_replay_v1',
 'execution_mode': 'paper_sim',
 'real_broker_orders': False,
 'artifacts': ['order_intents.csv', 'positions.csv', 'conclusion.md']}

## 中文结论

- 做了什么：验证 production paper/sim adapter contract，只记录 target-position intent。
- 核心指标：SOXL target_percent=0.25，paper quantity=50，cash basis=10000，price=50。
- regime 差异：本卡不做 regime 选择，只验证执行边界。
- 是否晋升：不代表策略晋升；这是 v1 paper/sim 执行边界卡。
- 风险：没有连接真实 broker，没有真实订单、成交、回报或账户状态。
- 下一步：接入 production strategy notebook 的固定参数、风险说明和 replay 输入输出。

paper/sim only；不触发真实 broker。